# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaitanyamadire01-dotcom/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Chosen Lane: Content Refresh Prioritization & CTR Opportunity Scoring (Predictive Ranking / Scoring).

Why this lane: Marketing and SEO content teams manage thousands of published URLs with finite editorial hours each quarter. Manually scanning Google Search Console reports causes teams to update pages that don't need fixing while genuinely declining high-traffic pages bleed impressions. Ranking pages by actionable decay and opportunity gives content teams a defensible weekly refresh queue.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Search Question: Which active client pages are suffering sustained performance decay (downward trend) and represent high recoverable search traffic if prioritized for an editorial refresh?

Unit of Analysis: A single URL/page per client aggregated over a 90-day rolling window (impressions_90d, clicks_90d, avg_position, trend_direction).

The Output: A ranked queue of URLs ordered by an opportunity score (or predicted probability of decline: P(is_declining_label = 1)) with primary reason codes (e.g., position loss vs. CTR drop).

The Action: The content team picks the top 20–50 URLs from this queue every sprint to update outdated statistics, refresh internal links, and rewrite meta titles/descriptions.

Cost of a Wrong Recommendation:

False Positive (recommending a stable page): Editorial hours (~2–5 hours per article) are wasted rewriting content that was already stable, introducing unnecessary risk of losing existing rankings.

False Negative (missing a decaying page): High-value organic positions continue to slide down the CTR cliff, permanently leaking traffic to competing domains.

Why Data/ML can help: A fixed hand-written threshold (e.g., "rewrite if position > 10") fails because traffic behaves non-linearly across position tiers and impression thresholds. Machine learning models capture multi-variable interactions across positions, impressions, and age to separate noise from true decay, consistently beating hand rules on Precision@50.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd

# Locate CSV
candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv. Check working directory.")

df = pd.read_csv(data_path)

# 1. Base rate of declining pages
total_pages = len(df)
declining_pages = df['trend_direction'].str.lower().eq('down').sum()
base_decay_rate = declining_pages / total_pages

# 2. Search volume vs impressions correlation
vol_imp_corr = df['search_volume'].corr(df['impressions_90d'])

# 3. Dynamic CTR breakdown across position tiers (for active pages with impressions >= 100)
active = df[df['impressions_90d'] >= 100]
ctr_by_tier = active.groupby('position_tier')['ctr'].mean().sort_values(ascending=False)

print(f"Total dataset size: {total_pages:,} pages")
print(f"1. Base Decay Rate (Pages labeled 'down'): {base_decay_rate:.1%} ({declining_pages:,} pages)")
print(f"2. Search Volume vs Impressions Correlation: {vol_imp_corr:.3f}")
print("\n3. CTR by Position Tier (Active Pages):")
for tier, val in ctr_by_tier.items():
    print(f"   - {tier}: {val:.2%}")

# Compute cliff between the top tier and the next visible tier
if len(ctr_by_tier) >= 2:
    top_tier, second_tier = ctr_by_tier.index[0], ctr_by_tier.index[1]
    cliff = ctr_by_tier.iloc[0] / ctr_by_tier.iloc[1]
    print(f"\nCTR drop from '{top_tier}' to '{second_tier}': {cliff:.1f}x cliff")

Total dataset size: 30,000 pages
1. Base Decay Rate (Pages labeled 'down'): 54.2% (16,262 pages)
2. Search Volume vs Impressions Correlation: 0.001

3. CTR by Position Tier (Active Pages):
   - page_1: 35.48%
   - top_3: 33.41%
   - striking: 25.58%
   - page_3_5: 14.24%
   - deep: 5.54%

CTR drop from 'page_1' to 'top_3': 1.1x cliff


### 3. Quick Look at the Data (Supporting Numbers)

Analysis of the 30,000-page starter dataset (`content_refresh_anonymized.csv`) demonstrates why an ML-driven prioritization queue is necessary:

1. **High Base Decay Rate (54.2%):** Out of 30,000 pages, 16,262 are actively decaying (`trend_direction == 'down'`). Because more than half the catalog is losing momentum, manual ad-hoc audits cannot scale; editorial teams need an algorithmic prioritization queue.
2. **Search Volume Decoupling (Correlation = 0.001):** The Pearson correlation between keyword search volume and actual 90-day impressions is essentially zero. Content teams prioritizing refreshes solely on high keyword search volume are chasing misleading proxies rather than actual traffic yield.
3. **The CTR Cliff Across Tiers:**
   - `page_1`: **35.48%**
   - `top_3`: **33.41%**
   - `striking` (positions 4–10): **25.58%**
   - `page_3_5`: **14.24%**
   - `deep`: **5.54%**
   
   A page slipping from Page 1 to deep search experiences a **6.4x collapse** in CTR, confirming that missing a declining page carries a severe organic traffic penalty.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I CAN claim:

"The model identifies observed statistical decay in past aggregated search impressions and CTR over 90-day intervals".

"The system provides an empirical decision-support queue that prioritizes pages with higher measured probability of traffic decline".

"Learned ranking rules achieve higher Precision@50 than simplistic single-variable heuristics on historical holdout client data".

What I CANNOT claim:

"The model decoded or reverse-engineered Google's proprietary search ranking algorithm".

"Updating these top 50 pages guarantees an immediate traffic increase or page-one ranking."

"Search volume dictates traffic yield" (disproved by the low correlation in Discovery A).

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Picked a provisional lane (Content Refresh & CTR Opportunity Scoring).
- [x] Defined the unit of analysis (client URL/page over 90 days).
- [x] Stated the decision, action, and cost of wrong calls.
- [x] Supported with 3 real figures from the dataset (54.2% decay rate, 0.001 volume correlation, 6.4x CTR cliff).
- [x] Added careful claims (decision-support, not reverse-engineering Google).